<a href="https://colab.research.google.com/github/yichungcheng/SSL_MSM_Endoscope_Spine_surgery/blob/main/00ssl_msn_v18_3_anticollapse_author_aligned_v16_base_hotfix_dataset_graphbool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSL-MSN V18.3 — anti-collapse + author-aligned pretraining regression build

本版由 V18.2 進版，目的不是單純修補 notebook，而是針對 V18.2 linear probe 未優於 V12 / random 的結果，回到 pretraining 訊號處理 partial collapse、prototype usage 不足與 teacher target 過度尖銳化問題。

設計原則：
- 以 v16 的 ViT-S、MSN loss、Sinkhorn-Knopp、me-max、gradient accumulation、checkpoint/export 架構為主體。
- 保留 V18.2 已修好的工程防呆：資料路徑、自動 image folder 搜尋、Keras 3 keyword call、masked positional embedding、restore/export smoke test、Colab 防浪費與 notebook regression scanner。
- V18.3 新增 anti-collapse 設計：提高 teacher temperature、放慢 teacher 更新、降低 me-max 主導性、提高 Sinkhorn 平滑與 iteration、降低 crop/mask 破壞性、加入 target uniform mix、加入 hard collapse early stop。
- 跨版本 checkpoint 一律只能 model-only warm start；正式 V18.3 預設 fresh start，避免把 V18.2 的 collapsed representation 帶入新實驗。

固定執行模式：
- `RUN_PREFLIGHT_ONLY`：只做 preflight / smoke test，不啟動訓練。
- `AUTO_START_TRAINING`：smoke test OK 後自動進入 `train_loop()`。
- `RUN_FINAL_EXPORT_AFTER_TRAIN`：訓練結束後自動 checkpoint/export/figures。


In [ ]:
# ============================================================
# 0. Import / install preflight — V18.3
# ============================================================
import sys
import subprocess
import importlib.util

REQUIRED_STDLIB_IMPORTS = [
    "os", "time", "json", "math", "random", "pathlib", "shutil",
    "glob", "csv", "re", "hashlib", "traceback", "ast"
]
for _mod in REQUIRED_STDLIB_IMPORTS:
    globals()[_mod] = __import__(_mod)

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TZ"] = "Asia/Taipei"
try:
    time.tzset()
except Exception:
    pass

import time
import json
import math
import random
import pathlib
import shutil
import glob
import csv
import re
import hashlib
import traceback
import ast
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"[OK] package available: {import_name}")

for _pkg in ["numpy", "pandas", "matplotlib", "tensorflow"]:
    ensure_package(_pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')

print("[PRECHECK] Python:", sys.version)
print("[PRECHECK] TensorFlow:", tf.__version__)
print("[PRECHECK] Keras:", tf.keras.__version__)
print("[PRECHECK] NumPy:", np.__version__)
print("[PRECHECK] Pandas:", pd.__version__)
print("[PRECHECK] Taipei time:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("[PRECHECK] GPUs:", gpus)
except Exception as e:
    print("[WARN] GPU memory growth setup skipped:", repr(e))


[OK] package available: numpy
[OK] package available: pandas
[OK] package available: matplotlib
[OK] package available: tensorflow
Mounted at /content/gdrive
[PRECHECK] Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[PRECHECK] TensorFlow: 2.20.0
[PRECHECK] Keras: 3.13.2
[PRECHECK] NumPy: 2.0.2
[PRECHECK] Pandas: 2.2.2
[PRECHECK] Taipei time: 2026-05-14 21:14:43
[PRECHECK] GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# import os, glob, shutil, json, re
# from pathlib import Path

# PRESERVE_EPOCHS = [6, 8, 9]

# RUN_NAME = "msn_vits_v18_3_anticollapse_author_aligned_v16_base"
# RUN_ROOT = "/content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/" + RUN_NAME

# CKPT_DIR = os.path.join(RUN_ROOT, "checkpoints")
# EXPORT_DIR = os.path.join(RUN_ROOT, "exports")
# FIGURE_DIR = os.path.join(RUN_ROOT, "figures")
# LOG_DIR = os.path.join(RUN_ROOT, "logs")

# PRESERVE_ROOT = os.path.join(RUN_ROOT, "preserved_checkpoints")
# os.makedirs(PRESERVE_ROOT, exist_ok=True)

# print("[PRESERVE ROOT]", PRESERVE_ROOT)
# print("[CKPT DIR]", CKPT_DIR)
# print("[EXPORT DIR]", EXPORT_DIR)

# def copy_file(src, dst_dir):
#     os.makedirs(dst_dir, exist_ok=True)
#     dst = os.path.join(dst_dir, os.path.basename(src))
#     shutil.copy2(src, dst)
#     print("[COPY]", src, "->", dst)

# def list_checkpoint_prefixes(ckpt_dir):
#     index_files = sorted(glob.glob(os.path.join(ckpt_dir, "ckpt-*.index")))
#     prefixes = [p[:-len(".index")] for p in index_files]
#     return prefixes

# ckpt_prefixes = list_checkpoint_prefixes(CKPT_DIR)

# print("\n[AVAILABLE CHECKPOINT PREFIXES]")
# for p in ckpt_prefixes:
#     print(" -", p)

# # 依照 V18.3 目前規則：每個 epoch 約 5000 global steps
# # epoch 6 -> ckpt-30000
# # epoch 8 -> ckpt-40000
# # epoch 9 -> ckpt-45000
# # 如果你的 steps_per_epoch 不同，下面會再用 summary/log 做補強搜尋。
# expected_global_steps = {
#     6: 30000,
#     8: 40000,
#     9: 45000,
# }

# for ep in PRESERVE_EPOCHS:
#     ep_dir = os.path.join(PRESERVE_ROOT, f"epoch_{ep:03d}")
#     os.makedirs(ep_dir, exist_ok=True)

#     print(f"\n========== Preserve epoch {ep} ==========")

#     # 1. Copy TF checkpoint by expected global_step
#     step = expected_global_steps.get(ep)
#     if step is not None:
#         prefix = os.path.join(CKPT_DIR, f"ckpt-{step}")
#         matched = glob.glob(prefix + ".*")
#         if matched:
#             ckpt_out = os.path.join(ep_dir, "tf_checkpoint")
#             os.makedirs(ckpt_out, exist_ok=True)

#             for f in matched:
#                 copy_file(f, ckpt_out)

#             # 建立該 epoch 自己的 checkpoint state file
#             with open(os.path.join(ckpt_out, "checkpoint"), "w", encoding="utf-8") as f:
#                 f.write(f'model_checkpoint_path: "ckpt-{step}"\n')
#                 f.write(f'all_model_checkpoint_paths: "ckpt-{step}"\n')

#             print(f"[OK] preserved TensorFlow checkpoint for epoch {ep}: ckpt-{step}")
#         else:
#             print(f"[WARN] expected checkpoint not found: ckpt-{step}")

#     # 2. Copy exported weights / manifests if filename contains epoch number
#     export_patterns = [
#         os.path.join(RUN_ROOT, "**", f"*epoch_{ep:03d}*"),
#         os.path.join(RUN_ROOT, "**", f"*epoch{ep:03d}*"),
#         os.path.join(RUN_ROOT, "**", f"*epoch_{ep:02d}*"),
#         os.path.join(RUN_ROOT, "**", f"*epoch{ep:02d}*"),
#         os.path.join(RUN_ROOT, "**", f"*epoch_{ep}*"),
#         os.path.join(RUN_ROOT, "**", f"*epoch{ep}*"),
#     ]

#     export_out = os.path.join(ep_dir, "exports_and_logs")
#     found_export = False

#     for pat in export_patterns:
#         for f in glob.glob(pat, recursive=True):
#             if os.path.isfile(f):
#                 # 避免把 preserved_checkpoints 自己又 copy 進去
#                 if "preserved_checkpoints" in f:
#                     continue
#                 copy_file(f, export_out)
#                 found_export = True

#     if not found_export:
#         print(f"[WARN] no epoch-pattern export/log found for epoch {ep}")

# print("\n[DONE] Epoch 6/8/9 preservation attempt completed.")
# print("[NEXT] Please inspect:", PRESERVE_ROOT)

[PRESERVE ROOT] /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/preserved_checkpoints
[CKPT DIR] /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkpoints
[EXPORT DIR] /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/exports

[AVAILABLE CHECKPOINT PREFIXES]
 - /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkpoints/ckpt-25000
 - /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkpoints/ckpt-30000
 - /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkpoints/ckpt-35000
 - /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkp

In [ ]:
# ============================================================
# 1. V18.3 config — v16 architecture preserved, v17.1 engineering safeguards, legacy restore safety added
# ============================================================
DRIVE_DIR = "/content/gdrive/MyDrive/PMBM/論文程式/MSM"
DATA_ROOT = os.path.join(DRIVE_DIR, "cholec80", "cholec80_extracted")
IMAGE_DIR = os.path.join(DATA_ROOT, "frames")

RUN_NAME = "msn_vits_v18_3_anticollapse_author_aligned_v16_base"
RUN_ROOT = os.path.join(DRIVE_DIR, "ssl_msn_paper_closer_tf", RUN_NAME)
CKPT_DIR = os.path.join(RUN_ROOT, "checkpoints")
EXPORT_DIR = os.path.join(RUN_ROOT, "exports")
LOG_DIR = os.path.join(RUN_ROOT, "logs")
REPORT_DIR = os.path.join(RUN_ROOT, "reports")
STATE_DIR = os.path.join(RUN_ROOT, "state")
INDEX_DIR = os.path.join(RUN_ROOT, "index")
DATASET_LOG_DIR = os.path.join(LOG_DIR, "dataset_loader")
FIGURE_DIR = os.path.join(RUN_ROOT, "figures")

for _d in [RUN_ROOT, CKPT_DIR, EXPORT_DIR, LOG_DIR, REPORT_DIR, STATE_DIR, INDEX_DIR, DATASET_LOG_DIR, FIGURE_DIR]:
    Path(_d).mkdir(parents=True, exist_ok=True)

# V18.3 default is fresh start. Previous runs are listed only for manual model-only warm start.
WARM_START_FROM_PREVIOUS_RUN = False  # V18.3 formal run defaults to fresh start; legacy runs are model-only warm-start only when manually enabled.
PREVIOUS_RUN_NAMES = [
    "msn_vits_v18_2_auto_training_mode_v16_base",
    "msn_vits_SSLMSNv16_auto_curriculum_full",
    "msn_vits_v17_1_anti_uniform_warmup_stable",
    "msn_vits_vSSLMSN15_9_robust_checkpoint_restore",
    "msn_vits_vSSLMSN15_8_checkpoint_thinning_best_model",
    "msn_vits_vSSLMSN15_7_disk_safe_checkpoint",
]
PREVIOUS_CKPT_DIRS = [os.path.join(DRIVE_DIR, "ssl_msn_paper_closer_tf", n, "checkpoints") for n in PREVIOUS_RUN_NAMES]

# Dataset / crop
IMG_SIZE = 224
PATCH_SIZE = 16
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]
IMAGE_EXTS_LOWER = [e.lower() for e in IMAGE_EXTS]
GLOBAL_CROP_SCALE = (0.60, 1.00)   # V18.3: less destructive global crop for medical/endoscopic frame semantics
LOCAL_CROP_SCALE = (0.35, 0.70)    # V18.3: avoid over-aggressive local crop that removes surgical context
BATCH_SIZE = 4
DROP_REMAINDER = True
SHUFFLE_BUFFER = 10000
TF_DATA_DETERMINISTIC = False
PREFETCH_BUFFER = tf.data.AUTOTUNE
AUTO_RESOLVE_IMAGE_DIR = True
DATASET_SEARCH_MAX_DEPTH = 7
DATASET_SEARCH_MAX_DIRS = 20000
DATASET_MIN_IMAGES_REQUIRED = 4
CHECK_BAD_IMAGES = False
ASSERT_DATA_FINITE = True
DETERMINISTIC_DATASET = False
RESHUFFLE_EACH_ITERATION = True

# ViT-S / MSN architecture — v16 retained
PROJECTION_DIM = 384
TRANSFORMER_LAYERS = 12
NUM_HEADS = 6
MLP_DIM = 1536
NUM_PROTOTYPES = 1024
PROJ_DIM = 256

# Views / masking — matches MSN ablation direction: 50% random masking + local views
NUM_GLOBAL_ANCHORS = 2
NUM_LOCAL_ANCHORS = 4
NUM_ANCHOR_VIEWS = NUM_GLOBAL_ANCHORS + NUM_LOCAL_ANCHORS
MASK_RATIO_GLOBAL = 0.40
MASK_RATIO_LOCAL = 0.30

# Training — v16 stable values retained
EPOCHS = 12
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
STUDENT_TEMP = 0.20
TEACHER_TEMP = 0.14
TEACHER_TEMP_WARMUP = True
TEACHER_TEMP_MIN = 0.12
TEACHER_TEMP_MAX = 0.18
TEACHER_TEMP_WARMUP_EPOCHS = 8
EMA_MOMENTUM = 0.998
LAMBDA_ME_MAX = 0.25
LAMBDA_ME_MAX_WARMUP = True
LAMBDA_ME_MAX_MIN = 0.00
LAMBDA_ME_MAX_MAX = 0.25
LAMBDA_ME_MAX_WARMUP_EPOCHS = 8
USE_SINKHORN = True
SINKHORN_EPS = 0.20
SINKHORN_ITERS = 6
EPS = 1e-8


# V18.3 anti-collapse controls
TARGET_UNIFORM_MIX = 0.02          # small uniform mixture prevents over-confident pseudo-targets without replacing Sinkhorn
MASK_RATIO_WARMUP = True
MASK_RATIO_GLOBAL_START = 0.25
MASK_RATIO_GLOBAL_END = MASK_RATIO_GLOBAL
MASK_RATIO_LOCAL_START = 0.15
MASK_RATIO_LOCAL_END = MASK_RATIO_LOCAL
MASK_RATIO_WARMUP_EPOCHS = 8
HARD_COLLAPSE_STOP = True
COLLAPSE_MIN_EPOCH = 4
COLLAPSE_PATIENCE_EPOCHS = 2
COLLAPSE_TEACHER_ENTROPY_MIN = 2.5
COLLAPSE_TEACHER_MAXP_MAX = 0.30
COLLAPSE_PROTO_H_MIN = 1.0
CHECKPOINT_SWEEP_EPOCHS = [2, 4, 6, 8, 10]

# v16 auto-curriculum retained
CURRICULUM_ENABLED = True
CURRICULUM_STAGES = [20000, 50000, 100000]
INITIAL_CURRICULUM_STAGE = 0
MAX_IMAGES = CURRICULUM_STAGES[INITIAL_CURRICULUM_STAGE]
CURRICULUM_MIN_STAGE_EPOCHS = 2
CURRICULUM_STABLE_EPOCHS_TO_ADVANCE = 2
CURRICULUM_CE_TARGET = 5.80
CURRICULUM_TEACHER_ENTROPY_MIN = 4.80
CURRICULUM_TEACHER_MAXP_MAX = 0.10

# Logging / checkpoint / Colab anti-waste
MAX_CKPT_KEEP = 30
SAVE_EVERY_EPOCH = 1
EXPORT_EVERY_EPOCH = 1
DEBUG_PRINT_EVERY = 100
MAX_LOG_FILE_MB = 20
MAX_WALLTIME_HOURS = 20.0
STOP_BEFORE_WALLTIME_MIN = 45
AUTO_STOP_ON_NO_PROGRESS = True
MIN_EPOCHS_BEFORE_STOP = 6
NO_PROGRESS_PATIENCE_EPOCHS = 5
MIN_QUALITY_DELTA = 0.25
TEACHER_MAXP_TOO_LOW = 0.003
TEACHER_MAXP_TOO_HIGH = 0.20
PROTO_H_TARGET_LOW = 3.0
LOSS_NAN_STOP = True
RUN_START_TS = time.time()


# Explicit execution mode — V18.3
# True: only run import/data/encoder/restore/export smoke tests and stop before training.
# False: enter training mode in the final launcher cell.
RUN_PREFLIGHT_ONLY = False
# True: when RUN_PREFLIGHT_ONLY=False, automatically start train_loop() after smoke tests.
# False: final cell prints a manual command instead of starting training.
AUTO_START_TRAINING = True
# True: after train_loop() exits normally or safely, run final_export_now(note="auto_after_train").
RUN_FINAL_EXPORT_AFTER_TRAIN = True

CONFIG_PATH = os.path.join(RUN_ROOT, "run_config_v18_3.json")
EPOCH_LOG_PATH = os.path.join(LOG_DIR, "epoch_summary.csv")
STEP_LOG_BASE = os.path.join(LOG_DIR, "step_metrics")
DECISION_LOG_PATH = os.path.join(LOG_DIR, "training_decisions.jsonl")
HEARTBEAT_PATH = os.path.join(STATE_DIR, "heartbeat.json")
TRAINING_STATE_JSON = os.path.join(STATE_DIR, "training_state.json")
CURRICULUM_STATE_PATH = os.path.join(STATE_DIR, "curriculum_state.json")
RESUME_AUDIT_PATH = os.path.join(LOG_DIR, "resume_audit.jsonl")
DATASET_MANIFEST_PATH = os.path.join(INDEX_DIR, "dataset_manifest.json")
USED_IMAGE_INDEX_CSV = os.path.join(INDEX_DIR, "used_images.csv")
RUN_SUMMARY_PATH = os.path.join(REPORT_DIR, "run_summary.json")

CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool, list, tuple, type(None)))}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print("IMAGE_DIR =", IMAGE_DIR)
print("RUN_ROOT  =", RUN_ROOT)
print("CONFIG    =", CONFIG_PATH)


IMAGE_DIR = /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames
RUN_ROOT  = /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base
CONFIG    = /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/run_config_v18_3.json


In [ ]:
# ============================================================
# 2. Dataset path / image loading — regression-safe utilities
# ============================================================
def list_dir_preview(path, max_items=20):
    p = Path(path)
    print(f"[DIR] {p}")
    if not p.exists():
        print("  - does not exist")
        return []
    try:
        items = sorted([x.name + ("/" if x.is_dir() else "") for x in p.iterdir()])[:max_items]
        for item in items:
            print("  -", item)
        return items
    except Exception as e:
        print("  - cannot list:", repr(e))
        return []


def is_image_path(path):
    return Path(str(path)).suffix.lower() in IMAGE_EXTS_LOWER


def _glob_image_files(path, recursive=True, max_files=None):
    """Google Drive / Colab-safe image search.

    V18.3 originally used Path.rglob('*') + p.is_file(). On some mounted
    Google Drive folders this can return zero files even when subfolders are
    visible. This replacement searches each supported extension directly with
    glob patterns, including upper/lower-case variants, and then de-duplicates.
    """
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return []

    patterns = []
    for ext in IMAGE_EXTS_LOWER:
        variants = {ext.lower(), ext.upper(), ext.capitalize()}
        for e in variants:
            patterns.append(("**/*" if recursive else "*") + e)

    files = []
    seen = set()
    for pat in patterns:
        try:
            for p in path.glob(pat):
                s = str(p)
                if s not in seen:
                    seen.add(s)
                    files.append(s)
                    if max_files is not None and len(files) >= int(max_files):
                        return sorted(files)
        except Exception as e:
            print(f"[WARN] glob failed: root={path} pattern={pat} err={repr(e)}")
    return sorted(files)


def count_images_in_dir_quick(path, recursive=True, stop_after=None):
    return len(_glob_image_files(path, recursive=recursive, max_files=stop_after))


def candidate_image_dirs(data_root, image_dir):
    roots = []
    for p in [
        image_dir,
        data_root,
        Path(data_root).parent,
        Path(data_root).parent / "cholec80",
        Path(data_root).parent / "cholec80" / "cholec80_extracted",
    ]:
        p = Path(p)
        if p not in roots:
            roots.append(p)

    names = ["frames", "frame", "images", "image", "imgs", "img", "cholec80_extracted/frames", "cholec80/frames"]
    cands = []
    for root in roots:
        cands.append(root)
        for name in names:
            cands.append(root / name)

    seen, out = set(), []
    for c in cands:
        s = str(c)
        if s not in seen:
            seen.add(s)
            out.append(c)
    return out


def preview_nested_image_dirs(root, max_video_dirs=5, max_files_each=3):
    """Debug helper: show whether videoXX subfolders actually contain images."""
    root = Path(root)
    if not root.exists() or not root.is_dir():
        return
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir()][:max_video_dirs]
    for d in subdirs:
        imgs = _glob_image_files(d, recursive=False, max_files=max_files_each)
        print(f"[NESTED PREVIEW] {d.name}: {len(imgs)} direct image sample(s)")
        for f in imgs[:max_files_each]:
            print("  -", Path(f).name)


def auto_resolve_image_dir(data_root=DATA_ROOT, image_dir=IMAGE_DIR):
    print("[DATASET PREFLIGHT] requested IMAGE_DIR:", image_dir)
    list_dir_preview(image_dir)
    list_dir_preview(Path(image_dir).parent)
    preview_nested_image_dirs(image_dir)

    best = None
    best_count = 0
    for cand in candidate_image_dirs(data_root, image_dir):
        if not cand.exists() or not cand.is_dir():
            continue
        n = count_images_in_dir_quick(cand, recursive=True, stop_after=DATASET_MIN_IMAGES_REQUIRED)
        print(f"[CANDIDATE] {cand} -> at least {n} image(s)")
        if n > best_count:
            best, best_count = cand, n
        if n >= DATASET_MIN_IMAGES_REQUIRED:
            print("[DATASET] resolved IMAGE_DIR =", cand)
            return str(cand)

    search_roots = [Path(data_root), Path(data_root).parent, Path(data_root).parent / "cholec80"]
    visited = 0
    for root in search_roots:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if visited >= DATASET_SEARCH_MAX_DIRS:
                break
            if not d.is_dir():
                continue
            try:
                rel_depth = len(d.relative_to(root).parts)
            except Exception:
                rel_depth = 999
            if rel_depth > DATASET_SEARCH_MAX_DEPTH:
                continue
            visited += 1
            if d.name.lower() in {"frames", "frame", "images", "image", "imgs", "img"} or d.name.lower().startswith("video"):
                n = count_images_in_dir_quick(d, recursive=True, stop_after=DATASET_MIN_IMAGES_REQUIRED)
                print(f"[SEARCH] {d} -> at least {n} image(s)")
                if n >= DATASET_MIN_IMAGES_REQUIRED:
                    # If this is a videoXX folder, use its parent as IMAGE_DIR so all videos are included.
                    resolved = d.parent if d.name.lower().startswith("video") else d
                    print("[DATASET] resolved IMAGE_DIR =", resolved)
                    return str(resolved)
                if n > best_count:
                    best, best_count = d, n

    msg = [
        "No image files were found after regression-safe dataset path search.",
        f"Requested IMAGE_DIR: {image_dir}",
        f"DATA_ROOT: {data_root}",
        f"Supported extensions: {IMAGE_EXTS}",
        "Debug note: V18.3 now uses glob-based extension search; if this still reports 0, run a shell find command to verify actual filenames.",
        "Possible causes:",
        "1) Google Drive is not mounted or the path is different in this Colab session.",
        "2) Frames were not extracted yet, or are under another folder name.",
        "3) File extensions are unsupported or files are nested outside DATA_ROOT/cholec80/cholec80_extracted.",
        "4) The parent folder exists but contains only videos or metadata, not images.",
    ]
    print("\n".join(msg))
    raise FileNotFoundError("\n".join(msg))


def list_image_files(image_root, max_images=None):
    image_root = Path(image_root)
    if not image_root.exists():
        raise FileNotFoundError(f"IMAGE_DIR does not exist: {image_root}")

    files = _glob_image_files(image_root, recursive=True, max_files=max_images)
    if max_images is not None:
        files = files[:int(max_images)]

    if len(files) == 0:
        list_dir_preview(image_root)
        list_dir_preview(image_root.parent)
        preview_nested_image_dirs(image_root)
        raise FileNotFoundError(
            "IMAGE_DIR exists but no supported image files were found.\n"
            f"IMAGE_DIR={image_root}\n"
            f"Supported extensions={IMAGE_EXTS}\n"
            "Check whether frames are under DATA_ROOT/cholec80/cholec80_extracted/frames, "
            "whether Google Drive is mounted, and whether images are nested under another folder."
        )
    return files


def file_sha256_text(items):
    h = hashlib.sha256()
    for item in items:
        h.update(str(item).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


def build_image_index(image_dir, max_images=None):
    files = list_image_files(image_dir, max_images=max_images)
    df = pd.DataFrame({"path": files})
    df["suffix"] = df["path"].map(lambda x: Path(x).suffix.lower())
    df["sha_path"] = df["path"].map(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()[:16])
    df.to_csv(USED_IMAGE_INDEX_CSV, index=False)
    manifest = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "image_dir": str(image_dir),
        "max_images": None if max_images is None else int(max_images),
        "num_images": int(len(files)),
        "supported_exts": IMAGE_EXTS,
        "fingerprint": file_sha256_text(files),
        "index_csv": USED_IMAGE_INDEX_CSV,
    }
    with open(DATASET_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"[DATASET] images={len(files)} index={USED_IMAGE_INDEX_CSV}")
    print(f"[DATASET] manifest={DATASET_MANIFEST_PATH}")
    return df, manifest


def decode_and_resize(image_bytes):
    x = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    x = tf.image.convert_image_dtype(x, tf.float32)
    x = tf.ensure_shape(x, [None, None, 3])
    return x


def read_image(path):
    return decode_and_resize(tf.io.read_file(path))


def random_resized_crop(x, out_size, scale=(0.5, 1.0)):
    x = tf.convert_to_tensor(x, tf.float32)
    h = tf.shape(x)[0]
    w = tf.shape(x)[1]
    area = tf.cast(h * w, tf.float32)
    target_area = tf.random.uniform([], scale[0], scale[1]) * area
    aspect_ratio = tf.random.uniform([], 3.0 / 4.0, 4.0 / 3.0)
    crop_h = tf.cast(tf.round(tf.sqrt(target_area / aspect_ratio)), tf.int32)
    crop_w = tf.cast(tf.round(tf.sqrt(target_area * aspect_ratio)), tf.int32)
    crop_h = tf.minimum(tf.maximum(crop_h, 16), h)
    crop_w = tf.minimum(tf.maximum(crop_w, 16), w)
    offset_h = tf.random.uniform([], 0, tf.maximum(h - crop_h + 1, 1), dtype=tf.int32)
    offset_w = tf.random.uniform([], 0, tf.maximum(w - crop_w + 1, 1), dtype=tf.int32)
    x = tf.image.crop_to_bounding_box(x, offset_h, offset_w, crop_h, crop_w)
    x = tf.image.resize(x, (out_size, out_size), antialias=True)
    return tf.clip_by_value(x, 0.0, 1.0)


def random_gaussian_blur(x, p=0.5):
    if tf.random.uniform([]) > p:
        return x
    x4 = tf.expand_dims(x, 0)
    x4 = tf.nn.avg_pool2d(x4, ksize=3, strides=1, padding="SAME")
    return tf.squeeze(x4, 0)


def color_jitter(x, s=0.4):
    x = tf.image.random_brightness(x, 0.8 * s)
    x = tf.image.random_contrast(x, 1.0 - 0.8 * s, 1.0 + 0.8 * s)
    x = tf.image.random_saturation(x, 1.0 - 0.8 * s, 1.0 + 0.8 * s)
    x = tf.image.random_hue(x, 0.2 * s)
    return tf.clip_by_value(x, 0.0, 1.0)


def normalize_imagenet(x):
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    return (x - mean) / std


def global_view_augment(x):
    x = random_resized_crop(x, IMG_SIZE, GLOBAL_CROP_SCALE)
    x = tf.image.random_flip_left_right(x)
    x = color_jitter(x, s=0.4)
    x = random_gaussian_blur(x, p=0.5)
    x = normalize_imagenet(tf.clip_by_value(x, 0.0, 1.0))
    return x


def local_view_augment(x):
    x = random_resized_crop(x, IMG_SIZE, LOCAL_CROP_SCALE)
    x = tf.image.random_flip_left_right(x)
    x = color_jitter(x, s=0.4)
    x = random_gaussian_blur(x, p=0.1)
    x = normalize_imagenet(tf.clip_by_value(x, 0.0, 1.0))
    return x


def make_msn_views(path):
    x = read_image(path)
    target = global_view_augment(x)
    global_anchors = [global_view_augment(x) for _ in range(NUM_GLOBAL_ANCHORS)]
    local_anchors = [local_view_augment(x) for _ in range(NUM_LOCAL_ANCHORS)]
    anchors = tf.stack(global_anchors + local_anchors, axis=0)
    target = tf.ensure_shape(target, [IMG_SIZE, IMG_SIZE, 3])
    anchors = tf.ensure_shape(anchors, [NUM_ANCHOR_VIEWS, IMG_SIZE, IMG_SIZE, 3])
    if ASSERT_DATA_FINITE:
        tf.debugging.assert_all_finite(target, "target view has NaN/Inf")
        tf.debugging.assert_all_finite(anchors, "anchor views have NaN/Inf")
    return anchors, target


def build_dataset(image_dir, batch_size=BATCH_SIZE, max_images=None):
    used_df, manifest = build_image_index(image_dir, max_images=max_images)
    files = used_df["path"].astype(str).tolist()
    if DETERMINISTIC_DATASET:
        files = sorted(files)
    ds = tf.data.Dataset.from_tensor_slices(files)
    ds = ds.shuffle(min(len(files), SHUFFLE_BUFFER), seed=SEED, reshuffle_each_iteration=RESHUFFLE_EACH_ITERATION)
    ds = ds.map(make_msn_views, num_parallel_calls=tf.data.AUTOTUNE, deterministic=TF_DATA_DETERMINISTIC)
    try:
        ds = ds.ignore_errors()
    except AttributeError:
        ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(batch_size, drop_remainder=DROP_REMAINDER)
    ds = ds.prefetch(PREFETCH_BUFFER)
    return ds, len(files), manifest


def smoke_test_dataset(ds):
    got = False
    for anchors_batch, target_batch in ds.take(1):
        got = True
        print("[SMOKE] anchors_batch shape:", anchors_batch.shape)
        print("[SMOKE] target_batch  shape:", target_batch.shape)
        if anchors_batch.shape.rank != 5:
            raise ValueError(f"anchors_batch rank must be 5 [B,V,H,W,C], got {anchors_batch.shape}")
        if target_batch.shape.rank != 4:
            raise ValueError(f"target_batch rank must be 4 [B,H,W,C], got {target_batch.shape}")
        tf.debugging.assert_shapes([
            (anchors_batch, (BATCH_SIZE, NUM_ANCHOR_VIEWS, IMG_SIZE, IMG_SIZE, 3)),
            (target_batch, (BATCH_SIZE, IMG_SIZE, IMG_SIZE, 3)),
        ])
        tf.debugging.assert_all_finite(anchors_batch, "anchors_batch has NaN/Inf")
        tf.debugging.assert_all_finite(target_batch, "target_batch has NaN/Inf")
    if not got:
        raise RuntimeError(
            "tf.data produced 0 batch after map/ignore_errors/batch. "
            "Likely causes: all images failed decode, batch_size > image count with drop_remainder=True, "
            "or IMAGE_DIR points to invalid files."
        )
    print("[SMOKE] dataset OK")

IMAGE_DIR = auto_resolve_image_dir(DATA_ROOT, IMAGE_DIR) if AUTO_RESOLVE_IMAGE_DIR else IMAGE_DIR
train_dataset, num_images, dataset_manifest = build_dataset(IMAGE_DIR, batch_size=BATCH_SIZE, max_images=MAX_IMAGES)
steps_per_epoch = num_images // BATCH_SIZE
print("[DATASET] steps_per_epoch =", steps_per_epoch)
smoke_test_dataset(train_dataset)


[DATASET PREFLIGHT] requested IMAGE_DIR: /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames
[DIR] /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted/frames
  - video01/
  - video02/
  - video03/
  - video04/
  - video05/
  - video06/
  - video07/
  - video08/
  - video09/
  - video10/
  - video11/
  - video12/
  - video13/
  - video14/
  - video15/
  - video16/
  - video17/
  - video18/
  - video19/
  - video20/
[DIR] /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted
  - frames/
  - phase_annotations/
  - tool_annotations/
[NESTED PREVIEW] video01: 3 direct image sample(s)
  - video01_000736.png
  - video01_000756.png
  - video01_000758.png
[NESTED PREVIEW] video02: 3 direct image sample(s)
  - video02_001848.png
  - video02_001852.png
  - video02_001869.png
[NESTED PREVIEW] video03: 3 direct image sample(s)
  - video03_004853.png
  - video03_004855.png
  - video03_004867.png
[NESTED PREVIEW] video04: 3 direct image sample(s)
  - vid

In [ ]:
# ============================================================
# 3. ViT-S encoder — masked positional embedding regression fix
# ============================================================
class Patches(layers.Layer):
    def __init__(self, patch_size=16, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dim = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dim])


class MLPBlock(layers.Layer):
    def __init__(self, dim, mlp_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.fc1 = layers.Dense(mlp_dim, activation=tf.nn.gelu)
        self.drop1 = layers.Dropout(dropout)
        self.fc2 = layers.Dense(dim)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        x = self.fc1(x)
        x = self.drop1(x, training=training)
        x = self.fc2(x)
        x = self.drop2(x, training=training)
        return x


class TransformerBlock(layers.Layer):
    def __init__(self, dim, num_heads, mlp_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads, dropout=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = MLPBlock(dim, mlp_dim, dropout=dropout)

    def call(self, x, training=False):
        y = self.norm1(x)
        x = x + self.attn(y, y, training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        return x


class V18ViTEncoder(tf.keras.Model):
    def __init__(self, img_size=224, patch_size=16, projection_dim=384, num_heads=6, depth=12, mlp_dim=1536, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.projection_dim = projection_dim
        self.patches = Patches(patch_size=patch_size)
        self.patch_proj = layers.Dense(projection_dim)
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, self.num_patches + 1, projection_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )
        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, projection_dim),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True,
        )
        self.blocks = [TransformerBlock(projection_dim, num_heads, mlp_dim, dropout=dropout) for _ in range(depth)]
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def random_mask_patch_tokens_and_pos(self, patch_tokens, mask_ratio):
        """Apply random patch masking in a TensorFlow graph-safe way.

        V18.3 uses dynamic mask warmup, so mask_ratio can be a scalar tf.Tensor
        inside @tf.function. Do not branch on that Tensor with a Python boolean.
        """
        mask_ratio = tf.cast(mask_ratio, tf.float32)

        def no_mask():
            pos = self.pos_embed[:, 1:, :]
            return patch_tokens + pos

        def with_mask():
            batch_size = tf.shape(patch_tokens)[0]
            num_tokens = tf.shape(patch_tokens)[1]
            keep_tokens = tf.cast(
                tf.round(tf.cast(num_tokens, tf.float32) * (1.0 - mask_ratio)),
                tf.int32,
            )
            keep_tokens = tf.maximum(1, keep_tokens)
            noise = tf.random.uniform([batch_size, num_tokens])
            ids_shuffle = tf.argsort(noise, axis=1)
            ids_keep = ids_shuffle[:, :keep_tokens]
            batch_idx = tf.tile(tf.range(batch_size)[:, None], [1, keep_tokens])
            gather_idx = tf.stack([batch_idx, ids_keep], axis=-1)
            kept_tokens = tf.gather_nd(patch_tokens, gather_idx)
            # Required V18 regression fix: gather from a 2D position table, not with batch_dims=1.
            pos_table = tf.squeeze(self.pos_embed[:, 1:, :], axis=0)
            pos = tf.gather(pos_table, ids_keep, axis=0)
            return kept_tokens + pos

        return tf.cond(tf.less_equal(mask_ratio, 0.0), no_mask, with_mask)

    def call(self, images, mask_ratio=0.0, training=False):
        x = self.patches(images)
        x = self.patch_proj(x)
        x = self.random_mask_patch_tokens_and_pos(x, mask_ratio=mask_ratio)
        batch_size = tf.shape(x)[0]
        cls = tf.tile(self.cls_token + self.pos_embed[:, :1, :], [batch_size, 1, 1])
        x = tf.concat([cls, x], axis=1)
        for blk in self.blocks:
            x = blk(x, training=training)
        x = self.norm(x)
        return x[:, 0, :]


class ProjectionHead(tf.keras.Model):
    def __init__(self, hidden_dim=1024, out_dim=PROJ_DIM, **kwargs):
        super().__init__(**kwargs)
        self.fc1 = layers.Dense(hidden_dim, activation=tf.nn.gelu)
        self.fc2 = layers.Dense(out_dim)

    def call(self, x, temp=None, training=False):
        del temp  # temperature is applied to prototype logits, but keyword is accepted for Keras 3 call safety.
        z = self.fc1(x)
        z = self.fc2(z)
        return tf.math.l2_normalize(z, axis=-1)


class Prototypes(tf.keras.Model):
    def __init__(self, num_prototypes=NUM_PROTOTYPES, proj_dim=PROJ_DIM, **kwargs):
        super().__init__(**kwargs)
        self.q = self.add_weight(
            name="prototype_matrix",
            shape=(num_prototypes, proj_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )

    def call(self, z):
        q = tf.math.l2_normalize(self.q, axis=-1)
        return tf.matmul(z, q, transpose_b=True)


def build_encoder(name=None):
    model = V18ViTEncoder(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        projection_dim=PROJECTION_DIM,
        num_heads=NUM_HEADS,
        depth=TRANSFORMER_LAYERS,
        mlp_dim=MLP_DIM,
        name=name,
    )
    _ = model(tf.zeros([2, IMG_SIZE, IMG_SIZE, 3], tf.float32), mask_ratio=MASK_RATIO_GLOBAL, training=False)
    return model


def build_head(name=None):
    head = ProjectionHead(name=name)
    _ = head(tf.zeros([2, PROJECTION_DIM], tf.float32), temp=STUDENT_TEMP, training=False)
    return head


def build_prototypes(name=None):
    p = Prototypes(name=name)
    _ = p(tf.zeros([2, PROJ_DIM], tf.float32))
    return p


def smoke_test_encoder_masked_pos():
    enc = build_encoder(name="smoke_encoder")
    x = tf.zeros([2, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)
    y = enc(x, mask_ratio=MASK_RATIO_GLOBAL, training=False)
    if y.shape != (2, PROJECTION_DIM):
        raise ValueError(f"Encoder smoke test output shape wrong: {y.shape}")
    tf.debugging.assert_all_finite(y, "encoder smoke output has NaN/Inf")
    print("[SMOKE] encoder batch=2 mask_ratio=", MASK_RATIO_GLOBAL, "output=", y.shape)

smoke_test_encoder_masked_pos()


[SMOKE] encoder batch=2 mask_ratio= 0.4 output= (2, 384)


In [ ]:
# ============================================================
# 4. MSN loss — paper-safe: target stop-gradient + Sinkhorn + me-max
# ============================================================
def safe_l2_normalize(x, axis=-1, eps=EPS):
    denom = tf.maximum(tf.norm(tf.cast(x, tf.float32), axis=axis, keepdims=True), eps)
    return tf.cast(x, tf.float32) / denom


def safe_softmax(logits, axis=-1):
    logits = tf.cast(logits, tf.float32)
    logits = logits - tf.reduce_max(logits, axis=axis, keepdims=True)
    probs = tf.nn.softmax(logits, axis=axis)
    probs = tf.clip_by_value(probs, EPS, 1.0)
    probs = probs / tf.reduce_sum(probs, axis=axis, keepdims=True)
    return probs


def entropy_from_probs(p, axis=-1):
    p = tf.clip_by_value(tf.cast(p, tf.float32), EPS, 1.0)
    return -tf.reduce_sum(p * tf.math.log(p), axis=axis)


def sinkhorn_knopp_stable(logits, epsilon=SINKHORN_EPS, num_iters=SINKHORN_ITERS, eps=EPS):
    logits = tf.cast(logits, tf.float32)
    logits = logits - tf.reduce_max(logits, axis=-1, keepdims=True)
    q = tf.exp(logits / tf.cast(epsilon, tf.float32))
    q = tf.transpose(q)
    q = q / tf.maximum(tf.reduce_sum(q), eps)
    k = tf.cast(tf.shape(q)[0], tf.float32)
    b = tf.cast(tf.shape(q)[1], tf.float32)
    for _ in range(num_iters):
        q = q / tf.maximum(tf.reduce_sum(q, axis=1, keepdims=True), eps)
        q = q / k
        q = q / tf.maximum(tf.reduce_sum(q, axis=0, keepdims=True), eps)
        q = q / b
    q = q * b
    q = tf.transpose(q)
    q = tf.clip_by_value(q, eps, 1.0)
    q = q / tf.reduce_sum(q, axis=-1, keepdims=True)
    return q


def msn_loss(student_logits_list, teacher_logits, teacher_temp, lambda_me_max):
    teacher_scaled = teacher_logits / tf.cast(teacher_temp, tf.float32)
    if USE_SINKHORN:
        target_probs = sinkhorn_knopp_stable(teacher_scaled, epsilon=SINKHORN_EPS, num_iters=SINKHORN_ITERS)
    else:
        target_probs = safe_softmax(teacher_scaled, axis=-1)

    # V18.3 anti-collapse: keep MSN/Sinkhorn target, but add a very small uniform component.
    # This prevents a few prototypes from dominating early pseudo-targets when teacher entropy collapses.
    if TARGET_UNIFORM_MIX > 0:
        uniform = tf.ones_like(target_probs) / tf.cast(NUM_PROTOTYPES, tf.float32)
        target_probs = (1.0 - tf.cast(TARGET_UNIFORM_MIX, tf.float32)) * target_probs + tf.cast(TARGET_UNIFORM_MIX, tf.float32) * uniform
        target_probs = target_probs / tf.reduce_sum(target_probs, axis=-1, keepdims=True)
    target_probs = tf.stop_gradient(target_probs)

    ce_terms = []
    anchor_probs = []
    for logits in student_logits_list:
        probs = safe_softmax(logits / tf.cast(STUDENT_TEMP, tf.float32), axis=-1)
        anchor_probs.append(probs)
        ce = -tf.reduce_mean(tf.reduce_sum(target_probs * tf.math.log(tf.clip_by_value(probs, EPS, 1.0)), axis=-1))
        ce_terms.append(ce)
    ce_loss = tf.add_n(ce_terms) / float(len(ce_terms))

    anchors = tf.concat(anchor_probs, axis=0)
    mean_p = tf.reduce_mean(anchors, axis=0)
    mean_p = tf.clip_by_value(mean_p, EPS, 1.0)
    mean_p = mean_p / tf.reduce_sum(mean_p)
    memax_entropy = -tf.reduce_sum(mean_p * tf.math.log(mean_p))
    memax_loss = -memax_entropy
    total_loss = ce_loss + tf.cast(lambda_me_max, tf.float32) * memax_loss

    pred_class = tf.argmax(anchors, axis=-1)
    proto_hist = tf.cast(tf.math.bincount(pred_class, minlength=NUM_PROTOTYPES, maxlength=NUM_PROTOTYPES), tf.float32)
    proto_hist = proto_hist / tf.maximum(tf.reduce_sum(proto_hist), 1.0)
    proto_hist_entropy = -tf.reduce_sum(tf.clip_by_value(proto_hist, EPS, 1.0) * tf.math.log(tf.clip_by_value(proto_hist, EPS, 1.0)))

    metrics = {
        "total_loss": tf.cast(total_loss, tf.float32),
        "ce_loss": tf.cast(ce_loss, tf.float32),
        "memax_loss": tf.cast(memax_loss, tf.float32),
        "memax_entropy": tf.cast(memax_entropy, tf.float32),
        "teacher_entropy": tf.reduce_mean(entropy_from_probs(target_probs)),
        "teacher_max_prob_mean": tf.reduce_mean(tf.reduce_max(target_probs, axis=-1)),
        "anchor_entropy": tf.reduce_mean(entropy_from_probs(anchors)),
        "prototype_hist_entropy": tf.cast(proto_hist_entropy, tf.float32),
    }
    return total_loss, metrics


def teacher_temp_for_epoch(epoch):
    if not TEACHER_TEMP_WARMUP:
        return float(TEACHER_TEMP)
    r = min(1.0, max(0.0, (float(epoch) - 1.0) / max(1.0, TEACHER_TEMP_WARMUP_EPOCHS)))
    return float(TEACHER_TEMP_MIN + r * (TEACHER_TEMP_MAX - TEACHER_TEMP_MIN))


def lambda_me_max_for_epoch(epoch):
    if not LAMBDA_ME_MAX_WARMUP:
        return float(LAMBDA_ME_MAX)
    r = min(1.0, max(0.0, (float(epoch) - 1.0) / max(1.0, LAMBDA_ME_MAX_WARMUP_EPOCHS)))
    return float(LAMBDA_ME_MAX_MIN + r * (LAMBDA_ME_MAX_MAX - LAMBDA_ME_MAX_MIN))


def _linear_warmup(epoch, start, end, warmup_epochs, enabled=True):
    if not enabled:
        return float(end)
    r = min(1.0, max(0.0, (float(epoch) - 1.0) / max(1.0, float(warmup_epochs))))
    return float(start + r * (end - start))


def mask_ratios_for_epoch(epoch):
    return (
        _linear_warmup(epoch, MASK_RATIO_GLOBAL_START, MASK_RATIO_GLOBAL_END, MASK_RATIO_WARMUP_EPOCHS, MASK_RATIO_WARMUP),
        _linear_warmup(epoch, MASK_RATIO_LOCAL_START, MASK_RATIO_LOCAL_END, MASK_RATIO_WARMUP_EPOCHS, MASK_RATIO_WARMUP),
    )


def smoke_test_loss_and_keras_call():
    enc = build_encoder(name="call_smoke_encoder")
    student_head = build_head(name="call_smoke_student_head")
    teacher_head = build_head(name="call_smoke_teacher_head")
    proto = build_prototypes(name="call_smoke_proto")
    x = tf.zeros([2, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)
    s = enc(x, mask_ratio=MASK_RATIO_GLOBAL, training=False)
    t = enc(x, mask_ratio=0.0, training=False)
    s_z = student_head(s, temp=STUDENT_TEMP, training=False)
    t_z = teacher_head(t, temp=TEACHER_TEMP, training=False)
    total, metrics = msn_loss([proto(s_z), proto(s_z)], proto(t_z), TEACHER_TEMP, LAMBDA_ME_MAX)
    tf.debugging.assert_all_finite(total, "smoke loss NaN/Inf")
    print("[SMOKE] Keras 3 keyword call and loss OK:", float(total.numpy()), {k: float(v.numpy()) for k, v in metrics.items()})

smoke_test_loss_and_keras_call()


[SMOKE] Keras 3 keyword call and loss OK: 5.259372711181641 {'total_loss': 5.259372711181641, 'ce_loss': 6.980161666870117, 'memax_loss': -6.883156776428223, 'memax_entropy': 6.883156776428223, 'teacher_entropy': 6.931472301483154, 'teacher_max_prob_mean': 0.0009765625, 'anchor_entropy': 6.882104873657227, 'prototype_hist_entropy': 0.00018844356236513704}


In [ ]:
# ============================================================
# 5. Build models, checkpoint, restore, and export smoke utilities — V18.3 patched
# ============================================================
student_encoder = build_encoder(name="student_encoder")
teacher_encoder = build_encoder(name="teacher_encoder")
student_head = build_head(name="student_head")
teacher_head = build_head(name="teacher_head")
prototypes = build_prototypes(name="prototypes")

# initialize teacher from student before any restore; checkpoint restore may overwrite both.
teacher_encoder.set_weights(student_encoder.get_weights())
teacher_head.set_weights(student_head.get_weights())
teacher_encoder.trainable = False
teacher_head.trainable = False

try:
    optimizer = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
except Exception:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

train_vars = student_encoder.trainable_variables + student_head.trainable_variables + prototypes.trainable_variables
accum_grad_vars = [tf.Variable(tf.zeros_like(v, dtype=tf.float32), trainable=False, name=f"accum_grad_{i}") for i, v in enumerate(train_vars)]

global_step_var = tf.Variable(0, dtype=tf.int64, trainable=False, name="global_step")
epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False, name="epoch")
resume_step_in_epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False, name="resume_step_in_epoch")
optimizer_step_var = tf.Variable(0, dtype=tf.int64, trainable=False, name="optimizer_step")
accum_step_var = tf.Variable(0, dtype=tf.int64, trainable=False, name="accum_step")
curriculum_stage_var = tf.Variable(INITIAL_CURRICULUM_STAGE, dtype=tf.int64, trainable=False, name="curriculum_stage")

# Full V18.3 checkpoint: used only when the checkpoint was produced by the same model/run layout.
ckpt = tf.train.Checkpoint(
    student_encoder=student_encoder,
    teacher_encoder=teacher_encoder,
    student_head=student_head,
    teacher_head=teacher_head,
    prototypes=prototypes,
    optimizer=optimizer,
    global_step=global_step_var,
    epoch=epoch_var,
    resume_step_in_epoch=resume_step_in_epoch_var,
    optimizer_step=optimizer_step_var,
    accum_step=accum_step_var,
    accum_grad_vars=accum_grad_vars,
    curriculum_stage=curriculum_stage_var,
)
manager = tf.train.CheckpointManager(ckpt, CKPT_DIR, max_to_keep=MAX_CKPT_KEEP)

# Model-only checkpoint: used for V16/V17 legacy warm start or as fallback after shape mismatch.
# It intentionally excludes optimizer and accum_grad_vars, because those variables depend on the exact
# train_vars order and shape. Restoring them across V16 -> V18.3 can map an old (1, 1, 384)
# accumulator into the new positional embedding accumulator (1, 197, 384), causing the reported error.
model_only_ckpt = tf.train.Checkpoint(
    student_encoder=student_encoder,
    teacher_encoder=teacher_encoder,
    student_head=student_head,
    teacher_head=teacher_head,
    prototypes=prototypes,
    global_step=global_step_var,
    epoch=epoch_var,
    resume_step_in_epoch=resume_step_in_epoch_var,
    optimizer_step=optimizer_step_var,
    curriculum_stage=curriculum_stage_var,
)


def zero_accumulation_buffers():
    for ag in accum_grad_vars:
        ag.assign(tf.zeros_like(ag))
    accum_step_var.assign(0)


def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def append_jsonl(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def is_valid_tf_checkpoint_prefix(prefix):
    if prefix is None:
        return False
    return tf.io.gfile.exists(prefix + ".index") and len(tf.io.gfile.glob(prefix + ".data-*-of-*")) > 0


def checkpoint_step_from_prefix(prefix):
    m = re.search(r"ckpt-(\d+)$", str(prefix))
    return int(m.group(1)) if m else -1


def latest_valid_checkpoint_in_dir(ckpt_dir):
    if not ckpt_dir or not tf.io.gfile.isdir(ckpt_dir):
        return None
    candidates = []
    state_latest = tf.train.latest_checkpoint(ckpt_dir)
    if state_latest:
        candidates.append(state_latest)
    for index_path in tf.io.gfile.glob(os.path.join(ckpt_dir, "ckpt-*.index")):
        candidates.append(index_path[:-len(".index")])
    candidates = sorted(set(candidates), key=checkpoint_step_from_prefix, reverse=True)
    valid = [p for p in candidates if is_valid_tf_checkpoint_prefix(p)]
    if not valid and candidates:
        print("[CKPT WARNING] broken checkpoint candidates:", candidates[:5])
    return valid[0] if valid else None


def resolve_restore_checkpoint():
    current = latest_valid_checkpoint_in_dir(CKPT_DIR)
    if current:
        return current, "resume_current"
    if WARM_START_FROM_PREVIOUS_RUN:
        for prev_dir in PREVIOUS_CKPT_DIRS:
            prev = latest_valid_checkpoint_in_dir(prev_dir)
            if prev:
                return prev, "warm_start_previous"
    return None, "fresh_start"


def restore_model_only(restore_path, restore_mode, reason=""):
    status = model_only_ckpt.restore(restore_path)
    status.expect_partial()
    zero_accumulation_buffers()
    dummy = tf.zeros([2, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)
    s = student_encoder(dummy, mask_ratio=MASK_RATIO_GLOBAL, training=False)
    t = teacher_encoder(dummy, mask_ratio=0.0, training=False)
    tf.debugging.assert_all_finite(s, "student encoder output after model-only restore has NaN/Inf")
    tf.debugging.assert_all_finite(t, "teacher encoder output after model-only restore has NaN/Inf")
    event = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "event": "model_only_restore",
        "mode": restore_mode,
        "checkpoint": restore_path,
        "reason": reason,
        "epoch": int(epoch_var.numpy()),
        "global_step": int(global_step_var.numpy()),
        "note": "optimizer and gradient accumulation buffers were intentionally skipped",
    }
    append_jsonl(RESUME_AUDIT_PATH, event)
    print(f"[CKPT] model-only restore OK ({restore_mode}): {restore_path}")
    print("[CKPT] skipped optimizer + accum_grad_vars; accumulation buffers reset to zero")
    return restore_path, restore_mode + "_model_only"


def safe_restore_checkpoint(ckpt_obj):
    restore_path, restore_mode = resolve_restore_checkpoint()
    if not restore_path:
        return None, "fresh_start"

    # Legacy checkpoints are intentionally treated as model-only warm starts.
    if restore_mode == "warm_start_previous":
        try:
            return restore_model_only(restore_path, restore_mode, reason="legacy previous-run warm start")
        except Exception:
            print("[CKPT MODEL-ONLY RESTORE FAILED]", restore_path)
            print(traceback.format_exc())
            return None, "fresh_start_after_model_only_restore_failure"

    # Same-run checkpoints should normally support full restore, including optimizer and accum buffers.
    try:
        ckpt_obj.restore(restore_path).expect_partial()
        print(f"[CKPT] full restore OK ({restore_mode}): {restore_path}")
        return restore_path, restore_mode
    except Exception:
        # Most common when CKPT_DIR accidentally points to an older V16 run or a previous incompatible run.
        err = traceback.format_exc()
        print("[CKPT FULL RESTORE FAILED]", restore_path)
        print(err)
        print("[CKPT] fallback to model-only restore; optimizer and accum_grad_vars will be skipped")
        try:
            return restore_model_only(restore_path, restore_mode, reason="full restore failed; likely optimizer/accumulator or architecture mismatch")
        except Exception:
            print("[CKPT FALLBACK MODEL-ONLY RESTORE FAILED]", restore_path)
            print(traceback.format_exc())
            return None, "fresh_start_after_restore_failure"


def safe_checkpoint_save(note=""):
    prefix = manager.save(checkpoint_number=int(global_step_var.numpy()))
    manifest = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "checkpoint_prefix": prefix,
        "epoch": int(epoch_var.numpy()),
        "global_step": int(global_step_var.numpy()),
        "note": note,
        "contains": ["student_encoder", "teacher_encoder", "student_head", "teacher_head", "prototypes", "optimizer", "global_step", "epoch", "accum_grad_vars"],
    }
    write_json(prefix + "_manifest.json", manifest)
    print("[CKPT] saved:", prefix)
    return prefix, manifest


def save_weights_with_manifest(model, path, epoch, global_step):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(str(path))
    manifest = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "path": str(path),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "sha256": sha256_file(path),
    }
    write_json(str(path) + ".manifest.json", manifest)
    return str(path), manifest


def restore_smoke_test_encoder(weights_path):
    enc = build_encoder(name="fresh_restore_smoke_encoder")
    dummy = tf.zeros([2, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)
    _ = enc(dummy, mask_ratio=MASK_RATIO_GLOBAL, training=False)
    enc.load_weights(weights_path)
    y = enc(dummy, mask_ratio=MASK_RATIO_GLOBAL, training=False)
    tf.debugging.assert_all_finite(y, "restore smoke output has NaN/Inf")
    print("[SMOKE] fresh encoder load_weights OK:", weights_path, y.shape)
    return True


def export_artifacts(epoch=None, note=""):
    e = int(epoch if epoch is not None else epoch_var.numpy())
    gs = int(global_step_var.numpy())
    exported = {}
    exported["student_encoder"] = save_weights_with_manifest(student_encoder, os.path.join(EXPORT_DIR, f"student_encoder_epoch_{e:03d}_step_{gs:07d}.weights.h5"), e, gs)[1]
    exported["teacher_encoder"] = save_weights_with_manifest(teacher_encoder, os.path.join(EXPORT_DIR, f"teacher_encoder_epoch_{e:03d}_step_{gs:07d}.weights.h5"), e, gs)[1]
    exported["student_head"] = save_weights_with_manifest(student_head, os.path.join(EXPORT_DIR, f"student_head_epoch_{e:03d}_step_{gs:07d}.weights.h5"), e, gs)[1]
    exported["teacher_head"] = save_weights_with_manifest(teacher_head, os.path.join(EXPORT_DIR, f"teacher_head_epoch_{e:03d}_step_{gs:07d}.weights.h5"), e, gs)[1]
    exported["prototypes"] = save_weights_with_manifest(prototypes, os.path.join(EXPORT_DIR, f"prototypes_epoch_{e:03d}_step_{gs:07d}.weights.h5"), e, gs)[1]
    restore_smoke_test_encoder(exported["student_encoder"]["path"])
    manifest = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "epoch": e, "global_step": gs, "note": note, "exports": exported}
    write_json(os.path.join(EXPORT_DIR, f"export_manifest_epoch_{e:03d}_step_{gs:07d}.json"), manifest)
    return manifest

latest_ckpt, restore_mode = safe_restore_checkpoint(ckpt)
append_jsonl(RESUME_AUDIT_PATH, {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "event": restore_mode, "checkpoint": latest_ckpt})


[CKPT] full restore OK (resume_current): /content/gdrive/MyDrive/PMBM/論文程式/MSM/ssl_msn_paper_closer_tf/msn_vits_v18_3_anticollapse_author_aligned_v16_base/checkpoints/ckpt-45000


In [ ]:
# ============================================================
# 6. Train step, logging, curriculum, and Colab anti-waste
# ============================================================
@tf.function
def reset_accum_grad_vars():
    for ag in accum_grad_vars:
        ag.assign(tf.zeros_like(ag))
    accum_step_var.assign(0)


def add_to_accum_grad_vars(step_grads):
    for ag, g in zip(accum_grad_vars, step_grads):
        ag.assign_add(tf.cast(g, tf.float32))
    accum_step_var.assign_add(1)


def update_teacher(student_model, teacher_model, momentum=EMA_MOMENTUM):
    for sw, tw in zip(student_model.weights, teacher_model.weights):
        tw.assign(tf.cast(momentum, tw.dtype) * tw + (1.0 - tf.cast(momentum, tw.dtype)) * tf.cast(sw, tw.dtype))


def apply_accumulated_grads():
    denom = tf.cast(tf.maximum(accum_step_var, 1), tf.float32)
    mean_grads = [ag / denom for ag in accum_grad_vars]
    mean_grads, grad_norm = tf.clip_by_global_norm(mean_grads, CLIP_NORM)
    optimizer.apply_gradients(zip(mean_grads, train_vars))
    optimizer_step_var.assign_add(1)
    update_teacher(student_encoder, teacher_encoder, EMA_MOMENTUM)
    update_teacher(student_head, teacher_head, EMA_MOMENTUM)
    reset_accum_grad_vars()
    return tf.cast(grad_norm, tf.float32)


@tf.function
def train_step(anchors_batch, target_batch, teacher_temp, lambda_me_max, mask_ratio_global, mask_ratio_local):
    with tf.GradientTape() as tape:
        student_logits_list = []
        for i in range(NUM_ANCHOR_VIEWS):
            view = anchors_batch[:, i, :, :, :]
            mask_ratio = mask_ratio_global if i < NUM_GLOBAL_ANCHORS else mask_ratio_local
            s_feat = student_encoder(view, mask_ratio=mask_ratio, training=True)
            s_z = student_head(s_feat, temp=STUDENT_TEMP, training=True)
            s_z = safe_l2_normalize(s_z, axis=-1)
            student_logits_list.append(prototypes(s_z))
        t_feat = teacher_encoder(target_batch, mask_ratio=0.0, training=False)
        t_z = teacher_head(t_feat, temp=teacher_temp, training=False)
        t_z = safe_l2_normalize(t_z, axis=-1)
        teacher_logits = prototypes(t_z)
        total_loss, metrics = msn_loss(student_logits_list, teacher_logits, teacher_temp, lambda_me_max)
    grads = tape.gradient(total_loss, train_vars)
    safe_grads = []
    for g, v in zip(grads, train_vars):
        if g is None:
            safe_grads.append(tf.zeros_like(v, dtype=tf.float32))
        else:
            g = tf.cast(g, tf.float32)
            safe_grads.append(tf.where(tf.math.is_finite(g), g, tf.zeros_like(g)))
    metrics["batch_grad_norm"] = tf.linalg.global_norm(safe_grads)
    return safe_grads, metrics


def elapsed_hours():
    return (time.time() - RUN_START_TS) / 3600.0


def walltime_remaining_min():
    return MAX_WALLTIME_HOURS * 60.0 - elapsed_hours() * 60.0


def should_stop_for_walltime():
    return walltime_remaining_min() <= STOP_BEFORE_WALLTIME_MIN


def heartbeat(extra=None):
    obj = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_hours": elapsed_hours(),
        "walltime_remaining_min": walltime_remaining_min(),
        "run_name": RUN_NAME,
    }
    if extra:
        obj.update(extra)
    write_json(HEARTBEAT_PATH, obj)
    return obj


def quality_score_from_epoch_metrics(m):
    vals = [m.get("teacher_entropy", np.nan), m.get("anchor_entropy", np.nan), m.get("prototype_hist_entropy", np.nan), m.get("teacher_max_prob_mean", np.nan)]
    if not np.isfinite(vals).all():
        return -1e9
    teacher_h, anchor_h, proto_h, maxp = [float(x) for x in vals]
    score = -abs(teacher_h - 5.0) - 0.7 * abs(anchor_h - 5.0) + 1.5 * max(0.0, proto_h - PROTO_H_TARGET_LOW)
    if maxp < TEACHER_MAXP_TOO_LOW:
        score -= 2.0
    if maxp > TEACHER_MAXP_TOO_HIGH:
        score -= 2.0
    return float(score)


def no_progress_decision(epoch, quality, best_quality, no_progress_epochs):
    improved = quality > best_quality + MIN_QUALITY_DELTA
    if improved:
        best_quality = quality
        no_progress_epochs = 0
    else:
        no_progress_epochs += 1
    stop = AUTO_STOP_ON_NO_PROGRESS and epoch >= MIN_EPOCHS_BEFORE_STOP and no_progress_epochs >= NO_PROGRESS_PATIENCE_EPOCHS
    decision = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "epoch": int(epoch), "quality": float(quality), "best_quality": float(best_quality), "no_progress_epochs": int(no_progress_epochs), "stop": bool(stop)}
    append_jsonl(DECISION_LOG_PATH, decision)
    if stop:
        print("[EARLY STOP] representation quality has not improved:", decision)
    return best_quality, no_progress_epochs, stop


def rotating_step_log_path():
    idx = 0
    while True:
        path = f"{STEP_LOG_BASE}_{idx:03d}.csv"
        if not os.path.exists(path) or os.path.getsize(path) < MAX_LOG_FILE_MB * 1024 * 1024:
            return path
        idx += 1


def append_step_log(row):
    path = rotating_step_log_path()
    exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def append_epoch_log(row):
    exists = os.path.exists(EPOCH_LOG_PATH)
    with open(EPOCH_LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def aggregate_metric_rows(rows):
    out = {}
    if not rows:
        return out
    keys = [k for k in rows[0].keys() if isinstance(rows[0][k], (int, float, np.floating))]
    for k in keys:
        vals = [float(r[k]) for r in rows if np.isfinite(float(r[k]))]
        if vals:
            out[k] = float(np.mean(vals))
    return out


def curriculum_decision(summary, curr_stage):
    if not CURRICULUM_ENABLED:
        return curr_stage, False, "disabled"
    if curr_stage >= len(CURRICULUM_STAGES) - 1:
        return curr_stage, False, "already_final_stage"
    ready = (
        summary.get("ce_loss", 999.0) <= CURRICULUM_CE_TARGET
        and summary.get("teacher_entropy", 0.0) >= CURRICULUM_TEACHER_ENTROPY_MIN
        and summary.get("teacher_max_prob_mean", 999.0) <= CURRICULUM_TEACHER_MAXP_MAX
    )
    if ready:
        return curr_stage + 1, True, "ssl_signal_ready"
    return curr_stage, False, "not_ready"


def rebuild_training_dataset_for_stage(stage):
    global MAX_IMAGES, train_dataset, num_images, steps_per_epoch, dataset_manifest
    MAX_IMAGES = int(CURRICULUM_STAGES[int(stage)])
    train_dataset, num_images, dataset_manifest = build_dataset(IMAGE_DIR, batch_size=BATCH_SIZE, max_images=MAX_IMAGES)
    steps_per_epoch = num_images // BATCH_SIZE
    smoke_test_dataset(train_dataset)
    print(f"[CURRICULUM] rebuilt dataset stage={stage} MAX_IMAGES={MAX_IMAGES} steps_per_epoch={steps_per_epoch}")


def is_hard_collapse(summary):
    return (
        float(summary.get("teacher_entropy", 999.0)) < COLLAPSE_TEACHER_ENTROPY_MIN
        and float(summary.get("teacher_max_prob_mean", 0.0)) > COLLAPSE_TEACHER_MAXP_MAX
        and float(summary.get("prototype_hist_entropy", 999.0)) < COLLAPSE_PROTO_H_MIN
    )


def train_loop():
    best_quality = -1e9
    no_progress_epochs = 0
    collapse_epochs = 0
    history = []
    start_epoch = int(epoch_var.numpy()) + 1
    curr_stage = int(curriculum_stage_var.numpy())
    if curr_stage != INITIAL_CURRICULUM_STAGE:
        rebuild_training_dataset_for_stage(curr_stage)

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_var.assign(epoch)
        t_temp = teacher_temp_for_epoch(epoch)
        lam = lambda_me_max_for_epoch(epoch)
        mask_g, mask_l = mask_ratios_for_epoch(epoch)
        print(f"[EPOCH {epoch}/{EPOCHS}] teacher_temp={t_temp:.4f} lambda_me_max={lam:.4f} mask_g={mask_g:.3f} mask_l={mask_l:.3f} MAX_IMAGES={MAX_IMAGES}")
        heartbeat({"epoch": epoch, "status": "epoch_start"})
        metric_rows = []
        reset_accum_grad_vars()

        for step, (anchors_batch, target_batch) in enumerate(train_dataset, start=1):
            if should_stop_for_walltime():
                print("[WALLTIME] approaching Colab limit; saving checkpoint and stopping safely.")
                safe_checkpoint_save(note="walltime_stop")
                export_artifacts(epoch=epoch, note="walltime_stop")
                write_json(TRAINING_STATE_JSON, {"epoch": epoch, "step": step, "global_step": int(global_step_var.numpy()), "reason": "walltime_stop"})
                return history

            grads, metrics = train_step(anchors_batch, target_batch, tf.constant(t_temp, tf.float32), tf.constant(lam, tf.float32), tf.constant(mask_g, tf.float32), tf.constant(mask_l, tf.float32))
            add_to_accum_grad_vars(grads)
            applied_grad_norm = np.nan
            if int(accum_step_var.numpy()) >= GRAD_ACCUM_STEPS:
                applied_grad_norm = float(apply_accumulated_grads().numpy())
            global_step_var.assign_add(1)

            row = {k: float(v.numpy()) for k, v in metrics.items()}
            row.update({"epoch": epoch, "step": step, "global_step": int(global_step_var.numpy()), "teacher_temp": t_temp, "lambda_me_max": lam, "mask_ratio_global": mask_g, "mask_ratio_local": mask_l, "applied_grad_norm": applied_grad_norm})
            metric_rows.append(row)
            if LOSS_NAN_STOP and not np.isfinite(row["total_loss"]):
                safe_checkpoint_save(note="nan_loss_stop")
                raise FloatingPointError("Loss became NaN/Inf; checkpoint saved.")
            if step % DEBUG_PRINT_EVERY == 0:
                append_step_log(row)
                print(f"epoch={epoch} step={step} loss={row['total_loss']:.4f} ce={row['ce_loss']:.4f} H={row['teacher_entropy']:.4f} maxp={row['teacher_max_prob_mean']:.5f} protoH={row['prototype_hist_entropy']:.4f} grad={row['batch_grad_norm']:.4f}")

        if int(accum_step_var.numpy()) > 0:
            apply_accumulated_grads()

        summary = aggregate_metric_rows(metric_rows)
        summary.update({"epoch": epoch, "global_step": int(global_step_var.numpy()), "max_images": int(MAX_IMAGES), "teacher_temp": t_temp, "lambda_me_max": lam, "mask_ratio_global": mask_g, "mask_ratio_local": mask_l})
        append_epoch_log(summary)
        history.append(summary)
        write_json(RUN_SUMMARY_PATH, history)
        print("[EPOCH SUMMARY]", json.dumps(summary, indent=2, ensure_ascii=False))

        if epoch % SAVE_EVERY_EPOCH == 0:
            safe_checkpoint_save(note="epoch_end")
        if epoch % EXPORT_EVERY_EPOCH == 0:
            export_artifacts(epoch=epoch, note="epoch_end")

        quality = quality_score_from_epoch_metrics(summary)
        best_quality, no_progress_epochs, stop_no_progress = no_progress_decision(epoch, quality, best_quality, no_progress_epochs)
        if stop_no_progress:
            safe_checkpoint_save(note="no_progress_stop")
            export_artifacts(epoch=epoch, note="no_progress_stop")
            break

        if HARD_COLLAPSE_STOP and epoch >= COLLAPSE_MIN_EPOCH and is_hard_collapse(summary):
            collapse_epochs += 1
            warning = {"epoch": epoch, "collapse_epochs": collapse_epochs, "summary": summary}
            append_jsonl(DECISION_LOG_PATH, {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "hard_collapse_warning", **warning})
            print("[COLLAPSE WARNING] teacher entropy / max_prob / prototype usage indicate hard partial collapse:", json.dumps(warning, ensure_ascii=False))
            if collapse_epochs >= COLLAPSE_PATIENCE_EPOCHS:
                safe_checkpoint_save(note="hard_collapse_stop")
                export_artifacts(epoch=epoch, note="hard_collapse_stop")
                print("[EARLY STOP] hard collapse persisted; stop training for parameter review.")
                break
        else:
            collapse_epochs = 0

        if epoch in CHECKPOINT_SWEEP_EPOCHS:
            append_jsonl(DECISION_LOG_PATH, {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "event": "checkpoint_sweep_candidate", "epoch": int(epoch), "export_dir": EXPORT_DIR})
            print(f"[SWEEP CANDIDATE] epoch {epoch} exported for downstream quick linear probe.")

        new_stage, advanced, reason = curriculum_decision(summary, curr_stage)
        append_jsonl(CURRICULUM_STATE_PATH + ".jsonl", {"epoch": epoch, "stage_before": curr_stage, "stage_after": new_stage, "advanced": advanced, "reason": reason})
        if advanced:
            curr_stage = new_stage
            curriculum_stage_var.assign(curr_stage)
            rebuild_training_dataset_for_stage(curr_stage)
            safe_checkpoint_save(note="curriculum_advanced")

    return history

print("[READY] Call train_loop() to start V18.3 training.")
# history = train_loop()


[READY] Call train_loop() to start V18.3 training.


In [ ]:
# ============================================================
# 7. Diagnostics figures and final export helper
# ============================================================
def load_epoch_history():
    if not os.path.exists(EPOCH_LOG_PATH):
        return pd.DataFrame()
    return pd.read_csv(EPOCH_LOG_PATH)


def make_paper_figures():
    df = load_epoch_history()
    if df.empty:
        print("[FIGURE] no epoch history yet:", EPOCH_LOG_PATH)
        return {}
    paths = {}
    for col in ["total_loss", "ce_loss", "memax_loss", "teacher_entropy", "teacher_max_prob_mean", "prototype_hist_entropy", "batch_grad_norm", "mask_ratio_global", "mask_ratio_local"]:
        if col not in df.columns:
            continue
        plt.figure(figsize=(7, 4))
        plt.plot(df["epoch"], df[col], marker="o")
        plt.xlabel("epoch")
        plt.ylabel(col)
        plt.title(f"V18.3 {col}")
        plt.grid(True, alpha=0.3)
        out = os.path.join(FIGURE_DIR, f"v18_3_{col}.png")
        plt.savefig(out, dpi=160, bbox_inches="tight")
        plt.close()
        paths[col] = out
    write_json(os.path.join(FIGURE_DIR, "figure_manifest.json"), paths)
    print("[FIGURE] saved:", paths)
    return paths


def final_export_now(note="manual_final_export"):
    safe_checkpoint_save(note=note)
    manifest = export_artifacts(epoch=int(epoch_var.numpy()), note=note)
    make_paper_figures()
    print("[FINAL EXPORT]", json.dumps(manifest, indent=2, ensure_ascii=False))
    return manifest

print("[READY] Call final_export_now() after training or before Colab shutdown.")


[READY] Call final_export_now() after training or before Colab shutdown.


In [ ]:
# ============================================================
# 8. Notebook final regression scanner — V18.3
# ============================================================
def forbidden_snippets():
    # Strings are assembled to avoid the scanner matching its own test cases.
    return [
        "tf.gather(self.pos_embed[:, 1:, :]" + ", ids, batch_dims=1)",
        "student_head(s," + " STUDENT_TEMP",
        "teacher_head(s," + " TEACHER_TEMP",
        "assert len(files) > 0, " + "f\"No image files found",
    ]


def validate_notebook_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if "cells" not in obj or not isinstance(obj["cells"], list):
        raise ValueError("notebook JSON does not contain a valid cells list")
    print("[VALIDATE] notebook JSON OK:", path)
    return obj


def validate_code_cell_syntax(path):
    obj = validate_notebook_json(path)
    errors = []
    for i, cell in enumerate(obj["cells"]):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        try:
            ast.parse(src)
        except SyntaxError as e:
            errors.append((i, str(e)))
    if errors:
        for i, e in errors:
            print(f"[SYNTAX ERROR] cell={i}: {e}")
        raise SyntaxError(f"{len(errors)} code cell(s) contain Python syntax errors")
    print("[VALIDATE] all code cell Python syntax OK:", path)


def scan_for_forbidden_snippets(path):
    txt = Path(path).read_text(encoding="utf-8")
    found = [s for s in forbidden_snippets() if s in txt]
    if found:
        print("[FORBIDDEN] old snippets remain:")
        for s in found:
            print("  -", s)
        raise AssertionError("Forbidden old error snippets remain in notebook")
    print("[VALIDATE] no forbidden old snippets found:", path)


def final_regression_check(notebook_path):
    validate_code_cell_syntax(notebook_path)
    scan_for_forbidden_snippets(notebook_path)
    print("[FINAL CHECK] notebook regression checks passed:", notebook_path)

# In Colab after upload/copy: final_regression_check("/content/.../ssl_msn_v18_3_auto_training_mode_v16_base.ipynb")


In [ ]:
# ============================================================
# 9. V18.3 execution launcher — explicit mode control
# ============================================================
if RUN_PREFLIGHT_ONLY:
    print("[MODE] preflight only. Training is not started.")
else:
    print("[MODE] training mode.")
    if AUTO_START_TRAINING:
        history = train_loop()
        if RUN_FINAL_EXPORT_AFTER_TRAIN:
            final_export_now(note="auto_after_train")
    else:
        print("[READY] AUTO_START_TRAINING=False. Run history = train_loop() manually.")


[MODE] training mode.
[EPOCH 10/12] teacher_temp=0.1800 lambda_me_max=0.2500 mask_g=0.400 mask_l=0.300 MAX_IMAGES=20000
epoch=10 step=100 loss=5.1935 ce=6.9257 H=6.2424 maxp=0.00344 protoH=2.0343 grad=1.2567
epoch=10 step=200 loss=5.2006 ce=6.9320 H=6.0934 maxp=0.00376 protoH=1.3410 grad=1.0435
epoch=10 step=300 loss=5.1986 ce=6.9309 H=6.4855 maxp=0.00291 protoH=2.3042 grad=0.2581
epoch=10 step=400 loss=5.1562 ce=6.8889 H=6.4695 maxp=0.00311 protoH=2.7056 grad=0.3781
epoch=10 step=500 loss=5.1192 ce=6.8504 H=6.2225 maxp=0.00349 protoH=2.2205 grad=0.9984
epoch=10 step=600 loss=5.1667 ce=6.8990 H=6.6538 maxp=0.00256 protoH=2.6119 grad=0.3910
epoch=10 step=700 loss=5.1521 ce=6.8843 H=6.3814 maxp=0.00316 protoH=2.6366 grad=0.4594
epoch=10 step=800 loss=5.2469 ce=6.9794 H=6.7501 maxp=0.00202 protoH=3.0627 grad=0.4555
epoch=10 step=900 loss=5.1536 ce=6.8860 H=6.6110 maxp=0.00263 protoH=2.8317 grad=0.6404
epoch=10 step=1000 loss=5.2219 ce=6.9545 H=6.6357 maxp=0.00264 protoH=2.6943 grad=0.3368